# Data Loading
The dataset is loaded using pandas library to begin preprocessing.

In [1]:
# Import pandas library
import pandas as pd

# Load dataset
df = pd.read_csv("skincare_products_clean.csv")

# Show first 5 rows
df.head()

,product_name,product_url,product_type,clean_ingreds,price
0,The Ordinary Natural Moisturising Factors + HA...,https://www.lookfantastic.com/the-ordinary-nat...,Moisturiser,"['capric triglyceride', 'cetyl alcohol', 'prop...",£5.20
1,CeraVe Facial Moisturising Lotion SPF 25 52ml,https://www.lookfantastic.com/cerave-facial-mo...,Moisturiser,"['homosalate', 'glycerin', 'octocrylene', 'eth...",£13.00
2,The Ordinary Hyaluronic Acid 2% + B5 Hydration...,https://www.lookfantastic.com/the-ordinary-hya...,Moisturiser,"['sodium hyaluronate', 'sodium hyaluronate', '...",£6.20
3,AMELIORATE Transforming Body Lotion 200ml,https://www.lookfantastic.com/ameliorate-trans...,Moisturiser,"['ammonium lactate', 'c12-15', 'glycerin', 'pr...",£22.50
4,CeraVe Moisturising Cream 454g,https://www.lookfantastic.com/cerave-moisturis...,Moisturiser,"['glycerin', 'cetearyl alcohol', 'capric trigl...",£16.00


# Dataset Information
This step is used to understand the dataset structure, columns, and data types.

In [2]:
# Display dataset information
print(df.info())

# Display column names
print(df.columns)

<class 'pandas.DataFrame'>
RangeIndex: 1138 entries, 0 to 1137
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_name   1138 non-null   str  
 1   product_url    1138 non-null   str  
 2   product_type   1138 non-null   str  
 3   clean_ingreds  1138 non-null   str  
 4   price          1138 non-null   str  
dtypes: str(5)
memory usage: 779.0 KB
None
Index(['product_name', 'product_url', 'product_type', 'clean_ingreds',
       'price'],
      dtype='str')


# Data Cleaning
Missing values and duplicate records are removed to improve data quality.

In [3]:
# Check missing values
print(df.isnull().sum())

# Drop rows where ingredients are missing
df = df.dropna(subset=['clean_ingreds'])

# Remove duplicate rows
df = df.drop_duplicates()

product_name     0
product_url      0
product_type     0
clean_ingreds    0
price            0
dtype: int64


# Column Renaming
The column 'clean_ingreds' is renamed to 'ingredients' for better readability.

In [4]:
# Rename column
df.rename(columns={'clean_ingreds': 'ingredients'}, inplace=True)

# Price Conversion
Special characters are removed from the price column and converted into numeric format.

In [5]:
# Remove special characters like ₹ and commas
df['price'] = df['price'].str.replace('[^0-9.]', '', regex=True)

# Convert to float
df['price'] = df['price'].astype(float)

# Text Standardization
Ingredient text is converted to lowercase for uniformity.

In [6]:
# Convert ingredients to lowercase
df['ingredients'] = df['ingredients'].str.lower()

# Ingredient Processing
Ingredient text is split, cleaned, and converted into a structured format.

In [7]:
# Convert ingredients string into list
df['ingredients_list'] = df['ingredients'].apply(lambda x: x.split(','))

# Remove extra spaces
df['ingredients_list'] = df['ingredients_list'].apply(
    lambda x: [i.strip() for i in x]
)

# Remove special characters
import re

def clean_ingredient(text):
    return re.sub(r'[^a-zA-Z\s]', '', text)

df['ingredients_list'] = df['ingredients_list'].apply(
    lambda x: [clean_ingredient(i) for i in x]
)

# Clean Ingredient Text
Ingredients are joined into a single clean string for TF-IDF processing.

In [8]:
# Join cleaned ingredients into single string
df['clean_ingredients'] = df['ingredients_list'].apply(
    lambda x: ' '.join(x)
)

# TF-IDF Processing
TF-IDF is applied to convert ingredient text into numerical features.

In [9]:
# Import TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF
tfidf = TfidfVectorizer()

# Transform ingredient text
tfidf_matrix = tfidf.fit_transform(df['clean_ingredients'])

# Check shape
print(tfidf_matrix.shape)

(1138, 2380)


# Save Final Dataset
The cleaned and processed dataset is saved for further model development.

In [10]:
# Save cleaned dataset
df.to_csv("final_clean_data.csv", index=False)